# 04 Behavioral Features: IEEE-CIS Fraud Detection

Это `Jupyter notebook` для следующего маленького шага после `03_baseline_model.ipynb`.

Цель:
- вынести новые feature-эксперименты в отдельный notebook;
- не перегружать baseline notebook;
- проверить, помогают ли простые behavioral и temporal признаки для anti-fraud;
- двигаться маленькими шагами и сохранять честную валидацию.


## План работы

1. Загрузить `train_transaction` и `train_identity`.
2. Собрать базовую таблицу для следующего feature-эксперимента.
3. Добавить `1` новый behavioral / temporal feature.
4. Сравнить результат с текущим честным baseline.
5. Коротко зафиксировать, что реально улучшилось, а что нет.


In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)


In [3]:
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
DOCS_DIR = PROJECT_ROOT / 'docs'
TRANSACTION_PATH = DATA_DIR / 'train_transaction.csv'
IDENTITY_PATH = DATA_DIR / 'train_identity.csv'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TRANSACTION_PATH exists:', TRANSACTION_PATH.exists())
print('IDENTITY_PATH exists:', IDENTITY_PATH.exists())


PROJECT_ROOT: /Users/drhtka/Downloads/Projects/Llm_ml_RAG/anti_fraud_analytics_platform
TRANSACTION_PATH exists: True
IDENTITY_PATH exists: True


In [4]:
def load_csv_if_exists(path: Path):
    if path.exists():
        print(f'Loaded: {path.name}')
        return pd.read_csv(path)
    print(f'File not found: {path}')
    return None


transactions = load_csv_if_exists(TRANSACTION_PATH)
identity = load_csv_if_exists(IDENTITY_PATH)


Loaded: train_transaction.csv
Loaded: train_identity.csv


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

## Первый temporal behavioral feature

На этом шаге мы не меняем алгоритм и не строим сложную архитектуру.

Мы проверяем одну маленькую гипотезу:

- если по одному `card1` транзакции идут слишком близко друг к другу,
- это может быть полезным anti-fraud сигналом,
- и такой сигнал можно превратить в признак для baseline модели.

Что именно мы изучаем:

- как отсортировать события по времени;
- как найти предыдущую транзакцию внутри одной сущности;
- как посчитать временной gap;
- как добавить temporal feature без лишнего усложнения.

Почему это хороший MVP-шаг:

- это только `1` новый признак;
- его легко объяснить;
- он ближе к реальной anti-fraud логике, чем еще один статический флаг.

Для текущего этапа берем `card1` как proxy entity.
Это не идеальная business entity, но этого достаточно, чтобы понять механику temporal feature engineering.

## Hypothesis: Fast Repeat Transaction Flag

На этом шаге проверяем не сырой temporal gap, а более понятный anti-fraud flag.

Гипотеза:

- если по одному `card1` следующая транзакция приходит меньше чем через `60` секунд,
- это может быть suspicious pattern,
- и такой бинарный feature может быть полезнее для baseline, чем непрерывный временной gap.

Что мы изучаем:

- как превратить temporal signal в понятный business flag;
- помогает ли rule-like feature улучшить baseline;
- меняется ли `precision`, `recall` и `manual_review_rate`.

Почему это хороший MVP-шаг:

- это только `1` новый признак;
- его легко объяснить;
- он ближе к реальной anti-fraud логике и ручным правилам.

In [6]:
from IPython.display import Markdown, display

high_risk_p_domains = {'outlook.com'}
high_risk_r_domains = {'outlook.com', 'icloud.com', 'gmail.com'}

feature_df = transactions[
    [
        'TransactionID',
        'isFraud',
        'TransactionAmt',
        'ProductCD',
        'card1',
        'card4',
        'card6',
        'P_emaildomain',
        'R_emaildomain',
        'addr1',
        'TransactionDT',
    ]
].copy()

feature_df['feat_productcd_c_flag'] = (feature_df['ProductCD'] == 'C').astype(int)
feature_df['feat_card4_discover_flag'] = (feature_df['card4'] == 'discover').astype(int)
feature_df['feat_card6_credit_flag'] = (feature_df['card6'] == 'credit').astype(int)
feature_df['feat_high_risk_p_email_flag'] = feature_df['P_emaildomain'].isin(high_risk_p_domains).astype(int)
feature_df['feat_high_risk_r_email_flag'] = feature_df['R_emaildomain'].isin(high_risk_r_domains).astype(int)
feature_df['feat_missing_r_email_flag'] = feature_df['R_emaildomain'].isna().astype(int)
feature_df['feat_amount_log1p'] = np.log1p(feature_df['TransactionAmt'])

base_feature_cols = [
    'feat_productcd_c_flag',
    'feat_high_risk_r_email_flag',
    'feat_card6_credit_flag',
    'feat_high_risk_p_email_flag',
    'feat_card4_discover_flag',
    'feat_missing_r_email_flag',
    'feat_amount_log1p',
]

train_df, valid_df = train_test_split(
    feature_df,
    test_size=0.2,
    random_state=42,
    stratify=feature_df['isFraud'],
)

train_df = train_df.copy()
valid_df = valid_df.copy()

card1_amount_stats = (
    train_df.groupby('card1')['TransactionAmt']
    .agg(card1_amt_mean='mean', card1_amt_std='std')
)
card1_txn_count = train_df.groupby('card1').size().rename('card1_txn_count')

train_df = train_df.join(card1_amount_stats, on='card1').join(card1_txn_count, on='card1').copy()
valid_df = valid_df.join(card1_amount_stats, on='card1').join(card1_txn_count, on='card1').copy()

for current_df in (train_df, valid_df):
    current_threshold = current_df['card1_amt_mean'] + 3 * current_df['card1_amt_std'].fillna(0)
    current_df['feat_amount_gt_card1_avg_plus_3std'] = (
        current_df['TransactionAmt'] > current_threshold
    ).astype(int)
    current_df['feat_card1_txn_count_log1p'] = np.log1p(current_df['card1_txn_count'].fillna(0))

def add_prev_txn_under_60s_flag(df: pd.DataFrame) -> pd.DataFrame:
    result = df.sort_values(['card1', 'TransactionDT', 'TransactionID']).copy()
    prev_dt = result.groupby('card1')['TransactionDT'].shift(1)
    result['time_since_prev_txn_by_card1'] = (result['TransactionDT'] - prev_dt).clip(lower=0)
    result['feat_card1_prev_txn_under_60s_flag'] = (
        result['time_since_prev_txn_by_card1'].fillna(np.inf) < 60
    ).astype(int)
    return result.sort_index()


train_df = add_prev_txn_under_60s_flag(train_df)
valid_df = add_prev_txn_under_60s_flag(valid_df)

y_train = train_df['isFraud'].copy()
y_valid = valid_df['isFraud'].copy()

feature_cols = base_feature_cols + [
    'feat_amount_gt_card1_avg_plus_3std',
    'feat_card1_txn_count_log1p',
    'feat_card1_prev_txn_under_60s_flag',
]

X_train = train_df[feature_cols].copy()
X_valid = valid_df[feature_cols].copy()

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('train fraud rate:', round(100 * y_train.mean(), 3), '%')
print('valid fraud rate:', round(100 * y_valid.mean(), 3), '%')

baseline_model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight='balanced',
)

baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_valid)
y_proba = baseline_model.predict_proba(X_valid)[:, 1]

metrics_summary = pd.DataFrame([
    {
        'precision': precision_score(y_valid, y_pred, zero_division=0),
        'recall': recall_score(y_valid, y_pred, zero_division=0),
        'f1': f1_score(y_valid, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_valid, y_proba),
    }
])

threshold_rows = []

for threshold in [0.3, 0.5, 0.7]:
    y_pred_threshold = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, y_pred_threshold).ravel()

    threshold_rows.append({
        'threshold': threshold,
        'precision': precision_score(y_valid, y_pred_threshold, zero_division=0),
        'recall': recall_score(y_valid, y_pred_threshold, zero_division=0),
        'f1': f1_score(y_valid, y_pred_threshold, zero_division=0),
        'predicted_fraud_count': int(y_pred_threshold.sum()),
        'manual_review_rate_pct': round(100 * y_pred_threshold.mean(), 2),
        'tp': int(tp),
        'fp': int(fp),
        'fn': int(fn),
        'tn': int(tn),
    })

threshold_df = pd.DataFrame(threshold_rows)

coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': baseline_model.coef_[0],
    'abs_coefficient': np.abs(baseline_model.coef_[0]),
}).sort_values('abs_coefficient', ascending=False)

reference_metrics = pd.DataFrame([
    {
        'model_version': '04_current_temporal_gap',
        'precision': 0.087325,
        'recall': 0.610452,
        'f1': 0.152793,
        'roc_auc': 0.744241,
    },
    {
        'model_version': '04_prev_txn_under_60s_flag',
        'precision': metrics_summary.loc[0, 'precision'],
        'recall': metrics_summary.loc[0, 'recall'],
        'f1': metrics_summary.loc[0, 'f1'],
        'roc_auc': metrics_summary.loc[0, 'roc_auc'],
    },
])

comparison_df = reference_metrics.set_index('model_version').T.copy()
comparison_df['delta_new_minus_reference'] = (
    comparison_df['04_prev_txn_under_60s_flag'] - comparison_df['04_current_temporal_gap']
)

display(Markdown("### metrics_summary"))
display(metrics_summary)

display(Markdown("### threshold_df"))
display(threshold_df)

display(Markdown("### coef_df"))
display(coef_df)

display(Markdown("### reference_metrics"))
display(reference_metrics)

display(Markdown("### comparison_df"))
display(comparison_df)


X_train: (472432, 10)
X_valid: (118108, 10)
train fraud rate: 3.499 %
valid fraud rate: 3.499 %


### metrics_summary

,precision,recall,f1,roc_auc
0,0.085936,0.612388,0.150722,0.746706


### threshold_df

,threshold,precision,recall,f1,predicted_fraud_count,manual_review_rate_pct,tp,fp,fn,tn
0,0.3,0.046611,0.878539,0.088525,77900,65.96,3631,74269,502,39706
1,0.5,0.085936,0.612388,0.150722,29452,24.94,2531,26921,1602,87054
2,0.7,0.141885,0.388338,0.207834,11312,9.58,1605,9707,2528,104268


...

,feature,coefficient,abs_coefficient
0,feat_productcd_c_flag,1.461134,1.461134
1,feat_high_risk_r_email_flag,0.987285,0.987285
2,feat_card6_credit_flag,0.626755,0.626755
4,feat_card4_discover_flag,0.561340,0.561340
9,feat_card1_prev_txn_under_60s_flag,0.446748,0.446748
7,feat_amount_gt_card1_avg_plus_3std,-0.354774,0.354774
6,feat_amount_log1p,0.349389,0.349389
3,feat_high_risk_p_email_flag,0.201865,0.201865
5,feat_missing_r_email_flag,-0.120123,0.120123
8,feat_card1_txn_count_log1p,0.046968,0.046968


### reference_metrics

,model_version,precision,recall,f1,roc_auc
0,04_current_temporal_gap,0.087325,0.610452,0.152793,0.744241
1,04_prev_txn_under_60s_flag,0.085936,0.612388,0.150722,0.746706


### comparison_df

model_version,04_current_temporal_gap,04_prev_txn_under_60s_flag,delta_new_minus_reference
precision,0.087325,0.085936,-0.001389
recall,0.610452,0.612388,0.001936
f1,0.152793,0.150722,-0.002071
roc_auc,0.744241,0.746706,0.002465


## Summary For Temporal Feature Experiment

### Что мы сделали

В этом notebook мы добавили `1` новый temporal behavioral feature:
`feat_time_since_prev_txn_by_card1_log1p`.

Он показывает, сколько времени прошло с предыдущей транзакции по тому же `card1`.

### Что важно понять

- temporal features ближе к реальной anti-fraud логике, чем просто статические флаги;
- даже полезный признак не обязан улучшать все метрики сразу;
- нужно смотреть не только на `roc_auc`, но и на `precision`, `recall` и `manual_review_rate`.

### Как интерпретировать результат

После запуска нужно ответить на вопросы:

1. вырос ли `recall`;
2. вырос ли `precision`;
3. изменился ли `manual_review_rate`;
4. стал ли temporal feature одним из заметных признаков по коэффициенту;
5. выглядит ли этот шаг полезным для следующей версии baseline.

### Короткий вывод

Если temporal feature дал только маленькое улучшение или смешанный trade-off, это нормальный результат.
Для MVP важно не “угадать идеальный признак”, а честно проверить гипотезу и понять, в какую сторону двигать feature engineering дальше.

### Interpretation

Добавление `feat_time_since_prev_txn_by_card1_log1p` дало смешанный результат.

Что улучшилось:
- `precision` немного вырос: `0.0852 -> 0.0873`
- `f1` немного вырос: `0.1497 -> 0.1528`
- `manual_review_rate` слегка снизился на рабочих threshold

Что ухудшилось:
- `recall` немного снизился: `0.6151 -> 0.6105`
- `roc_auc` тоже слегка снизился: `0.7466 -> 0.7442`

Вывод:
- temporal feature не дал явного качественного улучшения baseline;
- но эксперимент оказался полезным, потому что показал честный trade-off;
- в текущем виде этот feature не выглядит сильным драйвером модели и сам по себе не решает проблему false positives.